# 02_archetype_analysis.ipynb

名大会話コーパス (NUCC) を用いて、ユーザーの行動・発言パターンから「アーキタイプ（ペルソナ）」を抽出します。
これにより、参加者が会話の中でどのような役割（リーダー、フォロワー、盛り上げ役など）を果たしているかを分析します。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.loader.nucc_loader import NUCCLoader
from src.preprocessor.tokenizer import Tokenizer
from src.analysis.archetype import ArchetypeEngine

In [ ]:
# データの読み込み (NUCC)
data_path = Path.cwd().parent / 'data/raw/nucc/nucc'
loader = NUCCLoader(data_dir=str(data_path))
df = loader.load()

if df.empty:
    print("[ERROR] Failed to load NUCC data. Please check data/raw/nucc/nucc directory.")
else:
    print(f"Loaded {len(df)} records.")
    
    # 形態素解析
    # データ量が多い場合は処理時間を考慮
    print("Tokenizing...")
    tokenizer = Tokenizer()
    # デモ用にサンプリングしてもよいが、Archetype分析にはある程度の量が必要
    # df = df.sample(10000, random_state=42) 
    df['tokenized_text'] = df['text'].apply(lambda x: tokenizer.tokenize(x))
    print("Tokenization completed.")

## クラスタリングの実行
参加者を特徴量（発話頻度、ポジティブ/ネガティブ傾向、語彙の豊かさ）に基づいて4つのタイプに分類します。

In [ ]:
if not df.empty:
    engine = ArchetypeEngine(n_clusters=4)
    
    # 特徴量算出
    features_df = engine.analyze_user_characteristics(df)
    
    # アーキタイプ分類
    archetype_df = engine.classify_archetypes(features_df)
    
    print("Archetype Summary:")
    display(archetype_df['archetype_label'].value_counts())
    display(archetype_df.head(10))

## インタラクティブな可視化
各ユーザーがどのポジションにいるか（ポジティブかネガティブか、活動的か受動的か）を散布図で表現します。

In [ ]:
if not df.empty:
    fig = px.scatter(
        archetype_df.reset_index(), 
        x='msg_count', 
        y='avg_sentiment', 
        size='avg_char_len', 
        color='archetype_label',
        hover_name='user_id',
        title='User Archetype Map (NUCC)',
        labels={'msg_count': 'Activity Level (Msg Count)', 'avg_sentiment': 'Sentiment Score'},
        height=600
    )
    # 中心線の追加
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.show()